# ETL-13 · Chaves e duplicatas (02)

- **Etapa:** 02, chaves e duplicatas
- **Responsável:** Davi (@davi01mello)
- **Issue:** #15
- **Entrada:** `intermediarios/01_bruto.parquet` (saída da ETL-05)
- **Saída:** `intermediarios/02_chaves_duplicatas.parquet`

**O que esta etapa garante para o modelo estrela (ETL-10):**
1. Os IDs (`id_sinistro`, `id_pessoa`, `id_veiculo`) e o `cod_ibge` viram inteiros, sem perder nenhum valor.
2. Não há linhas 100% duplicadas. Se houver, são removidas e contadas.
3. `id_pessoa` é único, porque ele é a chave primária da fato (1 linha = 1 pessoa em 1 sinistro).
4. Cada `cod_ibge` tem um município só, porque ele é a chave natural da `dim_localidade`.

## 1. Setup

In [1]:
from pathlib import Path
import pandas as pd

# Raiz do repositório: sobe de pasta em pasta até achar a pasta .git
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".git").exists())

DADOS  = RAIZ / "data"                      # CSVs brutos (estão no Git)
INTERM = RAIZ / "data" / "intermediarios"   # .parquet (o .gitignore já ignora)
INTERM.mkdir(parents=True, exist_ok=True)

ENTRADA = INTERM / "01_bruto.parquet"       # na execução final, a saída da etapa anterior
SAIDA   = INTERM / "02_chaves_duplicatas.parquet"

COLUNAS_ID = ["id_sinistro", "id_pessoa", "id_veiculo", "cod_ibge"]
COLUNAS_RASTREIO = ["ano_origem", "arquivo_origem"]   # criadas na extração, não vêm da fonte

## 2. Carregar a entrada

Lê o `01_bruto.parquet` da ETL-05. **Enquanto ele não existir**, lê os 36 CSVs do mesmo jeito que a extração deve ler (tudo como texto, `latin-1`, `;`) e avisa. Quando a ETL-05 entrar na `main`, este bloco passa a usar o parquet sem nenhuma mudança.

In [2]:
def ler_csvs_brutos(pasta: Path) -> pd.DataFrame:
    """Provisório até a ETL-05: lê os 36 CSVs como texto e cria as colunas de rastreio."""
    partes = []
    for arq in sorted(pasta.glob("pessoas_*.csv")):
        d = pd.read_csv(arq, sep=";", encoding="latin-1", dtype=str,
                        keep_default_na=False, na_values=[""])
        d["ano_origem"] = int(arq.stem[-4:])
        d["arquivo_origem"] = arq.name
        partes.append(d)
    return pd.concat(partes, ignore_index=True)

if ENTRADA.exists():
    df = pd.read_parquet(ENTRADA)
    print("entrada:", ENTRADA.name)
else:
    df = ler_csvs_brutos(DADOS)
    print("AVISO: 01_bruto.parquet não existe ainda; lendo os CSVs direto (provisório até a ETL-05)")

print("shape:", df.shape)
df[COLUNAS_ID].head()

AVISO: 01_bruto.parquet não existe ainda; lendo os CSVs direto (provisório até a ETL-05)
shape: (806977, 33)


,id_sinistro,id_pessoa,id_veiculo,cod_ibge
0,2240338,1979797,2874666,3552502
1,2240338,1979798,2874666,3552502
2,1447800,1284984,NaN,3550308
3,1783410,563279,2648403,3550308
4,1798856,687322,NaN,3547601


## 3. Diagnóstico ANTES

In [3]:
def diagnostico(df: pd.DataFrame) -> pd.DataFrame:
    """Tabela-resumo usada antes e depois: tipo, % de nulos e nº de valores distintos por coluna."""
    return pd.DataFrame({
        'dtype': df.dtypes.astype(str),
        'pct_nulo': (df.isna().mean() * 100).round(2),
        'n_distintos': df.nunique(),
    })

def diagnostico_chaves(df: pd.DataFrame) -> pd.Series:
    """Os números que esta etapa precisa garantir, num quadro só (usado antes e depois)."""
    colunas_fonte = [c for c in df.columns if c not in COLUNAS_RASTREIO]
    return pd.Series({
        "linhas": len(df),
        "linhas 100% duplicadas (31 colunas da fonte)": int(df.duplicated(subset=colunas_fonte).sum()),
        "id_pessoa repetidos": int(df["id_pessoa"].duplicated().sum()),
        "id_pessoa nulos": int(df["id_pessoa"].isna().sum()),
        "id_sinistro nulos": int(df["id_sinistro"].isna().sum()),
        "id_veiculo nulos": int(df["id_veiculo"].isna().sum()),
        "sinistros distintos": int(df["id_sinistro"].nunique()),
        "cod_ibge com mais de 1 município": int((df.groupby("cod_ibge")["municipio"].nunique() > 1).sum()),
        "município com mais de 1 cod_ibge": int((df.groupby("municipio")["cod_ibge"].nunique() > 1).sum()),
    })

antes = diagnostico(df)
chaves_antes = diagnostico_chaves(df)
display(antes.loc[COLUNAS_ID])
chaves_antes

,dtype,pct_nulo,n_distintos
id_sinistro,str,0.00,402695
id_pessoa,str,0.00,806977
id_veiculo,str,21.04,554294
cod_ibge,str,0.00,645


linhas                                          806977
linhas 100% duplicadas (31 colunas da fonte)         0
id_pessoa repetidos                                  0
id_pessoa nulos                                      0
id_sinistro nulos                                    0
id_veiculo nulos                                169754
sinistros distintos                             402695
cod_ibge com mais de 1 município                     0
município com mais de 1 cod_ibge                     0
dtype: int64

## 4. Transformações

### 4.1 IDs e `cod_ibge` para inteiro

`id_sinistro`, `id_pessoa` e `cod_ibge` nunca são nulos e viram `int64`. `id_veiculo` é nulo em cerca de 21% das linhas (pedestres e registros sem veículo) e vira `Int64`, o inteiro do pandas que aceita nulo. Sem isso ele viraria `float` (`123.0`).

A conversão usa `errors="raise"`: se algum valor não for número, o notebook para em vez de virar nulo em silêncio. Depois a função confere que nenhum nulo novo apareceu.

In [4]:
def converter_ids(df: pd.DataFrame) -> pd.DataFrame:
    """Converte IDs e cod_ibge para inteiro; falha se algum valor não for numérico ou se surgir nulo novo."""
    df = df.copy()
    nulos_antes = df[COLUNAS_ID].isna().sum()
    for col in ["id_sinistro", "id_pessoa", "cod_ibge"]:
        df[col] = pd.to_numeric(df[col], errors="raise").astype("int64")
    df["id_veiculo"] = pd.to_numeric(df["id_veiculo"], errors="raise").astype("Int64")
    nulos_depois = df[COLUNAS_ID].isna().sum()
    assert (nulos_antes == nulos_depois).all(), "a conversão criou nulos novos"
    return df

df = converter_ids(df)
print(df[COLUNAS_ID].dtypes)
print("nulos por coluna (iguais aos de antes):")
print(df[COLUNAS_ID].isna().sum())

id_sinistro    int64
id_pessoa      int64
id_veiculo     Int64
cod_ibge       int64
dtype: object
nulos por coluna (iguais aos de antes):
id_sinistro         0
id_pessoa           0
id_veiculo     169754
cod_ibge            0
dtype: int64


### 4.2 Linhas 100% duplicadas

A comparação usa **as 31 colunas da fonte**, sem `ano_origem` e `arquivo_origem`. Assim, uma mesma linha que aparecesse em dois arquivos mensais também contaria como duplicata. Fica a primeira ocorrência.

In [5]:
def remover_duplicatas(df: pd.DataFrame) -> pd.DataFrame:
    """Remove linhas idênticas nas 31 colunas da fonte, mantendo a primeira ocorrência."""
    colunas_fonte = [c for c in df.columns if c not in COLUNAS_RASTREIO]
    return df.drop_duplicates(subset=colunas_fonte, keep="first").reset_index(drop=True)

n_antes = len(df)
df = remover_duplicatas(df)
print(f"linhas: {n_antes} -> {len(df)} (removidas: {n_antes - len(df)}, motivo: linha 100% duplicada)")

linhas: 806977 -> 806977 (removidas: 0, motivo: linha 100% duplicada)


### 4.3 `id_pessoa` único

É a chave primária da fato. Se algum `id_pessoa` aparecer duas vezes com conteúdo diferente, o notebook para e mostra os casos, porque escolher uma das linhas seria uma decisão do grupo, não desta etapa.

In [6]:
def garantir_id_pessoa_unico(df: pd.DataFrame) -> pd.DataFrame:
    """Confere que id_pessoa é único; se não for, mostra os casos e interrompe."""
    repetidos = df[df["id_pessoa"].duplicated(keep=False)].sort_values("id_pessoa")
    if len(repetidos):
        display(repetidos)
        raise ValueError(f"{repetidos['id_pessoa'].nunique()} id_pessoa repetidos com conteúdo diferente")
    print(f"id_pessoa único: {df['id_pessoa'].nunique():,} valores em {len(df):,} linhas".replace(",", "."))
    return df

df = garantir_id_pessoa_unico(df)

id_pessoa único: 806.977 valores em 806.977 linhas


### 4.4 `cod_ibge` → município

Cada código IBGE deve ter um município só, e vice-versa, porque o `cod_ibge` é a chave natural da `dim_localidade`. Esta conferência também olha a região administrativa, já que cada município deve pertencer a uma região só.

In [7]:
def conferir_localidade(df: pd.DataFrame) -> pd.DataFrame:
    """Confere cod_ibge 1:1 com municipio e 1 região por município; devolve a tabela de municípios."""
    loc = df.groupby("cod_ibge").agg(
        municipios=("municipio", "nunique"),
        regioes=("regiao_administrativa", "nunique"),
        municipio=("municipio", "first"),
        regiao_administrativa=("regiao_administrativa", "first"),
        linhas=("id_pessoa", "size"),
    )
    problemas = loc[(loc["municipios"] > 1) | (loc["regioes"] > 1)]
    assert problemas.empty, f"{len(problemas)} cod_ibge com mais de um município ou região"
    assert df.groupby("municipio")["cod_ibge"].nunique().max() == 1, "município com mais de um cod_ibge"
    print(f"{len(loc)} códigos IBGE, {df['municipio'].nunique()} municípios, "
          f"{df['regiao_administrativa'].nunique()} regiões: relação 1:1 confirmada")
    return loc

localidades = conferir_localidade(df)
localidades.sort_values("linhas", ascending=False).head(10)

645 códigos IBGE, 645 municípios, 16 regiões: relação 1:1 confirmada


,municipios,regioes,municipio,regiao_administrativa,linhas
cod_ibge,,,,,
3550308,1,1,SAO PAULO,METROPOLITANA DE SÃO PAULO,186723
3543402,1,1,RIBEIRAO PRETO,RIBEIRÃO PRETO,21395
3509502,1,1,CAMPINAS,CAMPINAS,20429
3518800,1,1,GUARULHOS,METROPOLITANA DE SÃO PAULO,19452
3549805,1,1,SAO JOSE DO RIO PRETO,SÃO JOSÉ DO RIO PRETO,16521
3552205,1,1,SOROCABA,SOROCABA,14786
3548708,1,1,SAO BERNARDO DO CAMPO,METROPOLITANA DE SÃO PAULO,14494
3547809,1,1,SANTO ANDRE,METROPOLITANA DE SÃO PAULO,11767
3525904,1,1,JUNDIAI,CAMPINAS,11090


### 4.5 Conferências extras de integridade (só consulta)

Três perguntas que a ETL-23 (fato) vai precisar que sejam verdade:
- um veículo (`id_veiculo`) pertence a um sinistro só?
- um sinistro tem uma data e um município só?
- há pessoas com **tudo igual menos o `id_pessoa`**? Não são duplicatas (são pessoas diferentes), mas vale saber quantas são.

In [8]:
veiculo_em_varios = (df.dropna(subset=["id_veiculo"]).groupby("id_veiculo")["id_sinistro"].nunique() > 1).sum()
sinistro_varias_datas = (df.groupby("id_sinistro")["data_sinistro"].nunique() > 1).sum()
sinistro_varios_municipios = (df.groupby("id_sinistro")["cod_ibge"].nunique() > 1).sum()

colunas_sem_id = [c for c in df.columns if c not in ["id_pessoa", *COLUNAS_RASTREIO]]
quase = df[df.duplicated(subset=colunas_sem_id, keep=False)]

print("veículos em mais de 1 sinistro:", veiculo_em_varios)
print("sinistros com mais de 1 data:", sinistro_varias_datas)
print("sinistros com mais de 1 município:", sinistro_varios_municipios)
print(f"linhas iguais exceto id_pessoa: {len(quase)} em "
      f"{quase.groupby(colunas_sem_id, dropna=False).ngroups} grupos")
quase["tipo_de_vitima"].value_counts()

veículos em mais de 1 sinistro: 0
sinistros com mais de 1 data: 0
sinistros com mais de 1 município: 0
linhas iguais exceto id_pessoa: 1211 em 571 grupos


tipo_de_vitima
NAO DISPONIVEL    643
PASSAGEIRO        470
PEDESTRE           88
CONDUTOR           10
Name: count, dtype: int64

## 5. Diagnóstico DEPOIS (mesma tabela do passo 3)

In [9]:
depois = diagnostico(df)
chaves_depois = diagnostico_chaves(df)
display(depois.loc[COLUNAS_ID])
pd.DataFrame({"antes": chaves_antes, "depois": chaves_depois})

,dtype,pct_nulo,n_distintos
id_sinistro,int64,0.00,402695
id_pessoa,int64,0.00,806977
id_veiculo,Int64,21.04,554294
cod_ibge,int64,0.00,645


,antes,depois
linhas,806977,806977
linhas 100% duplicadas (31 colunas da fonte),0,0
id_pessoa repetidos,0,0
id_pessoa nulos,0,0
id_sinistro nulos,0,0
id_veiculo nulos,169754,169754
sinistros distintos,402695,402695
cod_ibge com mais de 1 município,0,0
município com mais de 1 cod_ibge,0,0


## 6. Salvar a saída

In [10]:
df.to_parquet(SAIDA, index=False)
print('salvo em:', SAIDA.relative_to(RAIZ), '| shape:', df.shape)

salvo em: data/intermediarios/02_chaves_duplicatas.parquet | shape: (806977, 33)


## 7. Resumo

- **IDs para inteiro:** `id_sinistro`, `id_pessoa` e `cod_ibge` viraram `int64`, e `id_veiculo` virou `Int64` (aceita nulo). Nenhum valor deixou de ser número e nenhum nulo novo apareceu. Os 21% de `id_veiculo` nulos continuam nulos, porque são pedestres e registros sem veículo.
- **Duplicatas:** **0** linhas 100% duplicadas nas 31 colunas da fonte, então nenhuma linha foi removida. A base sai com as mesmas 806.977 linhas.
- **`id_pessoa`:** é único nas 806.977 linhas e pode ser a chave primária da fato.
- **`cod_ibge`:** 645 códigos para 645 municípios (1:1), cada um em uma única das 16 regiões. A chave natural da `dim_localidade` é segura.
- **Integridade extra:** nenhum veículo aparece em mais de um sinistro, e cada sinistro tem uma data e um município só.
- **Achado:** 1.211 linhas (571 grupos) têm tudo igual exceto o `id_pessoa`. Não foram removidas, porque são pessoas diferentes no mesmo sinistro (a maioria passageiros ou registros com tipo de vítima "NAO DISPONIVEL"). O fato é registrado para a análise saber que existem.